In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import librosa

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "ESC-50-master"
    / "ESC-50-master"
)

META_PATH = (
    DATASET_PATH
    / "meta"
    / "esc50.csv"
)

AUDIO_PATH = (
    DATASET_PATH
    / "audio"
)

print(META_PATH.exists())
print(AUDIO_PATH.exists())

True
True


In [3]:
df = pd.read_csv(META_PATH)

natural_categories = [
    "chirping_birds",
    "crow",
    "crickets",
    "frog",
    "insects",
    "rain",
    "wind",
    "thunderstorm",
    "sea_waves",
    "water_drops",
    "pouring_water",
    "crackling_fire",
    "rooster"
]

natural_df = df[
    df["category"].isin(natural_categories)
]

print(natural_df.shape)

(520, 7)


In [4]:
def extract_mfcc_features(signal, sr):

    features = {}

    mfccs = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )

    for i in range(13):

        features[f"mfcc_{i+1}"] = np.mean(
            mfccs[i]
        )

    return features

In [5]:
sample_row = natural_df.iloc[0]

audio_file = AUDIO_PATH / sample_row["filename"]

signal, sr = librosa.load(
    audio_file,
    sr=None
)

test_features = extract_mfcc_features(
    signal,
    sr
)

test_features

{'mfcc_1': np.float32(-254.93631),
 'mfcc_2': np.float32(85.83962),
 'mfcc_3': np.float32(-107.10316),
 'mfcc_4': np.float32(31.011576),
 'mfcc_5': np.float32(-39.987736),
 'mfcc_6': np.float32(-17.290342),
 'mfcc_7': np.float32(-37.214104),
 'mfcc_8': np.float32(-18.14569),
 'mfcc_9': np.float32(-11.705071),
 'mfcc_10': np.float32(-17.621454),
 'mfcc_11': np.float32(-26.893171),
 'mfcc_12': np.float32(-13.461913),
 'mfcc_13': np.float32(-9.417187)}

In [6]:
all_features = []

for _, row in natural_df.iterrows():

    audio_file = AUDIO_PATH / row["filename"]

    signal, sr = librosa.load(
        audio_file,
        sr=None
    )

    feature_dict = extract_mfcc_features(
        signal,
        sr
    )

    feature_dict["filename"] = row["filename"]

    feature_dict["category"] = row["category"]

    all_features.append(
        feature_dict
    )

mfcc_df = pd.DataFrame(
    all_features
)

print(mfcc_df.shape)

mfcc_df.head()

(520, 15)


,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13,filename,category
0,-254.936310,85.839622,-107.103157,31.011576,-39.987736,-17.290342,-37.214104,-18.145691,-11.705071,-17.621454,-26.893171,-13.461913,-9.417187,1-100038-A-14.wav,chirping_birds
1,-466.622101,144.261398,22.961349,52.213543,2.970680,11.319603,-2.895933,0.076406,-8.929120,10.019114,-6.963048,-5.848204,-11.108883,1-101296-A-19.wav,thunderstorm
2,-441.774231,153.395920,32.532093,38.864769,0.386356,-1.514098,2.681722,-3.470331,-3.546912,5.873673,-2.664442,-7.838368,-6.315548,1-101296-B-19.wav,thunderstorm
3,-173.182404,131.663696,-23.350657,44.626896,-16.704086,34.943077,13.708267,24.017622,-3.575602,12.136042,-1.485329,13.711203,4.838860,1-103298-A-9.wav,crow
4,-400.280853,155.725174,73.404076,15.198190,5.499929,16.989422,18.179058,7.343544,-0.929693,-1.168285,0.959946,1.117658,0.381365,1-115521-A-19.wav,thunderstorm


In [7]:
mfcc_df.to_csv(
    PROJECT_ROOT
    / "outputs"
    / "mfcc_features.csv",
    index=False
)

print("MFCC Features Saved")

MFCC Features Saved


In [8]:
mfcc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   mfcc_1    520 non-null    float32
 1   mfcc_2    520 non-null    float32
 2   mfcc_3    520 non-null    float32
 3   mfcc_4    520 non-null    float32
 4   mfcc_5    520 non-null    float32
 5   mfcc_6    520 non-null    float32
 6   mfcc_7    520 non-null    float32
 7   mfcc_8    520 non-null    float32
 8   mfcc_9    520 non-null    float32
 9   mfcc_10   520 non-null    float32
 10  mfcc_11   520 non-null    float32
 11  mfcc_12   520 non-null    float32
 12  mfcc_13   520 non-null    float32
 13  filename  520 non-null    str    
 14  category  520 non-null    str    
dtypes: float32(13), str(2)
memory usage: 34.7 KB


In [9]:
def extract_advanced_features(signal, sr):

    features = {}

    # -------------------
    # MFCC
    # -------------------

    mfccs = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )

    for i in range(13):

        features[f"mfcc_{i+1}"] = np.mean(
            mfccs[i]
        )

    # -------------------
    # Chroma
    # -------------------

    chroma = librosa.feature.chroma_stft(
        y=signal,
        sr=sr
    )

    for i in range(12):

        features[f"chroma_{i+1}"] = np.mean(
            chroma[i]
        )

    # -------------------
    # Spectral Contrast
    # -------------------

    contrast = librosa.feature.spectral_contrast(
        y=signal,
        sr=sr
    )

    for i in range(contrast.shape[0]):

        features[f"contrast_{i+1}"] = np.mean(
            contrast[i]
        )

    # -------------------
    # Tonnetz
    # -------------------

    harmonic = librosa.effects.harmonic(
        signal
    )

    tonnetz = librosa.feature.tonnetz(
        y=harmonic,
        sr=sr
    )

    for i in range(tonnetz.shape[0]):

        features[f"tonnetz_{i+1}"] = np.mean(
            tonnetz[i]
        )

    return features

In [10]:
sample_row = natural_df.iloc[0]

audio_file = AUDIO_PATH / sample_row["filename"]

signal, sr = librosa.load(
    audio_file,
    sr=None
)

test_features = extract_advanced_features(
    signal,
    sr
)

print(len(test_features))

test_features

38


{'mfcc_1': np.float32(-254.93631),
 'mfcc_2': np.float32(85.83962),
 'mfcc_3': np.float32(-107.10316),
 'mfcc_4': np.float32(31.011576),
 'mfcc_5': np.float32(-39.987736),
 'mfcc_6': np.float32(-17.290342),
 'mfcc_7': np.float32(-37.214104),
 'mfcc_8': np.float32(-18.14569),
 'mfcc_9': np.float32(-11.705071),
 'mfcc_10': np.float32(-17.621454),
 'mfcc_11': np.float32(-26.893171),
 'mfcc_12': np.float32(-13.461913),
 'mfcc_13': np.float32(-9.417187),
 'chroma_1': np.float32(0.21310173),
 'chroma_2': np.float32(0.2549494),
 'chroma_3': np.float32(0.30068514),
 'chroma_4': np.float32(0.49204376),
 'chroma_5': np.float32(0.3580956),
 'chroma_6': np.float32(0.3707463),
 'chroma_7': np.float32(0.37274998),
 'chroma_8': np.float32(0.45591554),
 'chroma_9': np.float32(0.44967347),
 'chroma_10': np.float32(0.24055381),
 'chroma_11': np.float32(0.22109084),
 'chroma_12': np.float32(0.20356974),
 'contrast_1': np.float64(17.26835493979131),
 'contrast_2': np.float64(11.827682304875184),
 'contras

In [11]:
all_features = []

for _, row in natural_df.iterrows():

    audio_file = AUDIO_PATH / row["filename"]

    signal, sr = librosa.load(
        audio_file,
        sr=None
    )

    feature_dict = extract_advanced_features(
        signal,
        sr
    )

    feature_dict["filename"] = row["filename"]

    feature_dict["category"] = row["category"]

    all_features.append(
        feature_dict
    )

advanced_df = pd.DataFrame(
    all_features
)

print(advanced_df.shape)

advanced_df.head()

C:\Natural_Sound_Statistics_Project\venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


(520, 40)


,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,...,contrast_6,contrast_7,tonnetz_1,tonnetz_2,tonnetz_3,tonnetz_4,tonnetz_5,tonnetz_6,filename,category
0,-254.936310,85.839622,-107.103157,31.011576,-39.987736,-17.290342,-37.214104,-18.145691,-11.705071,-17.621454,...,20.464414,45.704715,-0.058836,0.004075,0.051868,0.035464,-0.014806,-0.015532,1-100038-A-14.wav,chirping_birds
1,-466.622101,144.261398,22.961349,52.213543,2.970680,11.319603,-2.895933,0.076406,-8.929120,10.019114,...,16.233288,28.473721,0.002136,-0.003342,-0.027320,0.071995,-0.018120,0.001736,1-101296-A-19.wav,thunderstorm
2,-441.774231,153.395920,32.532093,38.864769,0.386356,-1.514098,2.681722,-3.470331,-3.546912,5.873673,...,16.746074,28.574655,-0.005311,0.008006,-0.021965,0.031961,0.004954,0.005678,1-101296-B-19.wav,thunderstorm
3,-173.182404,131.663696,-23.350657,44.626896,-16.704086,34.943077,13.708267,24.017622,-3.575602,12.136042,...,15.456582,42.903741,0.002883,0.021413,-0.012975,-0.041927,0.009535,0.009389,1-103298-A-9.wav,crow
4,-400.280853,155.725174,73.404076,15.198190,5.499929,16.989422,18.179058,7.343544,-0.929693,-1.168285,...,15.181650,16.613661,0.004814,0.000623,-0.009786,0.010140,0.004016,0.000385,1-115521-A-19.wav,thunderstorm


In [12]:
advanced_df.to_csv(
    PROJECT_ROOT
    / "outputs"
    / "advanced_audio_features.csv",
    index=False
)

print("Advanced Features Saved")

Advanced Features Saved


In [13]:
advanced_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 40 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   mfcc_1      520 non-null    float32
 1   mfcc_2      520 non-null    float32
 2   mfcc_3      520 non-null    float32
 3   mfcc_4      520 non-null    float32
 4   mfcc_5      520 non-null    float32
 5   mfcc_6      520 non-null    float32
 6   mfcc_7      520 non-null    float32
 7   mfcc_8      520 non-null    float32
 8   mfcc_9      520 non-null    float32
 9   mfcc_10     520 non-null    float32
 10  mfcc_11     520 non-null    float32
 11  mfcc_12     520 non-null    float32
 12  mfcc_13     520 non-null    float32
 13  chroma_1    520 non-null    float32
 14  chroma_2    520 non-null    float32
 15  chroma_3    520 non-null    float32
 16  chroma_4    520 non-null    float32
 17  chroma_5    520 non-null    float32
 18  chroma_6    520 non-null    float32
 19  chroma_7    520 non-null    float32
 20 